# Advanced Push Pipelines with Generator Coroutines — Tutorial Problems and Solutions

In this notebook we will keep working with **push-based pipelines**.

Instead of asking a source for the next value, we will push values into a pipeline by calling:

```python
pipeline.send(value)
```

Each pipeline stage will:

1. wait for a value using `yield`;
2. process that value;
3. push zero, one, or many values into the next stage.

This notebook uses a tutorial style:

- every problem is divided into small logical steps;
- design decisions are explained before code is introduced;
- intermediate pipeline versions are tested;
- the final solution is built from reusable pieces;
- edge cases and failure modes are discussed.

All examples use only the Python standard library.


## 1. A Small Foundation

Before solving the advanced problems, we need a minimal set of tools.

The first tool is an auto-priming decorator.

A newly created generator is initially suspended before its first line of execution.  
Calling `next(generator)` advances it to the first `yield`, after which `.send(value)` can be used.


In [1]:
from __future__ import annotations

from collections import defaultdict, deque
from dataclasses import dataclass
from functools import wraps
from math import sqrt
from typing import Any, Callable, Deque, Dict, Generator, Hashable, Iterable, List, Mapping, Optional, Tuple


def autostart(function: Callable[..., Generator[Any, Any, Any]]):
    """Create and prime a generator coroutine."""
    @wraps(function)
    def wrapper(*args: Any, **kwargs: Any):
        generator = function(*args, **kwargs)
        next(generator)
        return generator
    return wrapper


### A sink for examples and tests

Printing is useful while learning, but storing results in a list is much easier to test.

The following sink receives values forever and appends them to a list.


In [2]:
@autostart
def list_sink(storage: List[Any]):
    while True:
        value = yield
        storage.append(value)


@autostart
def print_sink(label: str = ""):
    while True:
        value = yield
        print(f"{label}{value}")


### A source helper

A push pipeline still needs something that sends values into its first stage.

This helper is deliberately simple. It pushes every value and optionally closes the pipeline at the end.


In [3]:
def push_all(
    values: Iterable[Any],
    pipeline: Generator[Any, Any, Any],
    *,
    close_when_done: bool = True,
) -> None:
    for value in values:
        pipeline.send(value)

    if close_when_done:
        pipeline.close()


# Problem 1 — Parsing Delimited Text into Valid Records

Suppose we receive lines in the following format:

```text
timestamp|device|reading
```

Example:

```text
2026-01-01T10:00:00|sensor-A|21.5
```

We want a pipeline that:

1. splits each line;
2. verifies that exactly three fields are present;
3. converts the reading to `float`;
4. rejects malformed input;
5. stores valid records as dictionaries.

We will build this solution one step at a time.


## Step 1 — Define the target record

Using a dataclass makes the expected output structure explicit.


In [4]:
@dataclass(frozen=True)
class SensorRecord:
    timestamp: str
    device: str
    reading: float


## Step 2 — Write a strict parser

The parser is an ordinary function.

Keeping parsing logic outside the coroutine has two advantages:

- it can be tested independently;
- the coroutine remains focused on pipeline control.


In [5]:
def parse_sensor_line(line: str) -> SensorRecord:
    parts = [part.strip() for part in line.split("|")]

    if len(parts) != 3:
        raise ValueError(f"expected 3 fields, received {len(parts)}")

    timestamp, device, reading_text = parts

    if not timestamp:
        raise ValueError("timestamp is empty")
    if not device:
        raise ValueError("device is empty")

    return SensorRecord(
        timestamp=timestamp,
        device=device,
        reading=float(reading_text),
    )


## Step 3 — Decide how failures should travel

A production pipeline should not silently discard malformed values.

We will create a small error record containing:

- the original input;
- the exception type;
- the error message.


In [6]:
@dataclass(frozen=True)
class ParseFailure:
    raw_value: Any
    exception_type: str
    message: str


## Step 4 — Create a parsing stage

The parsing stage has two downstream targets:

- valid values go to `success_target`;
- failed values go to `failure_target`.

The stage handles only the exceptions we expect from bad input.  
Unexpected programming errors should still propagate.


In [7]:
@autostart
def parsing_stage(
    parser: Callable[[Any], Any],
    success_target: Generator[Any, Any, Any],
    failure_target: Generator[Any, Any, Any],
):
    try:
        while True:
            raw_value = yield

            try:
                parsed_value = parser(raw_value)
            except (ValueError, TypeError) as exc:
                failure_target.send(
                    ParseFailure(
                        raw_value=raw_value,
                        exception_type=type(exc).__name__,
                        message=str(exc),
                    )
                )
            else:
                success_target.send(parsed_value)
    finally:
        success_target.close()
        failure_target.close()


## Step 5 — Assemble and test the pipeline

Notice that the first stage receives raw strings, but the successful sink receives `SensorRecord` objects.


In [8]:
valid_sensor_records: List[SensorRecord] = []
invalid_sensor_lines: List[ParseFailure] = []

sensor_pipeline = parsing_stage(
    parse_sensor_line,
    list_sink(valid_sensor_records),
    list_sink(invalid_sensor_lines),
)

sensor_lines = [
    "2026-01-01T10:00:00|sensor-A|21.5",
    "2026-01-01T10:00:01|sensor-B|bad",
    "missing|fields",
    "2026-01-01T10:00:03|sensor-C|19.75",
]

push_all(sensor_lines, sensor_pipeline)

assert valid_sensor_records == [
    SensorRecord("2026-01-01T10:00:00", "sensor-A", 21.5),
    SensorRecord("2026-01-01T10:00:03", "sensor-C", 19.75),
]
assert len(invalid_sensor_lines) == 2

valid_sensor_records, invalid_sensor_lines


([SensorRecord(timestamp='2026-01-01T10:00:00', device='sensor-A', reading=21.5),
  SensorRecord(timestamp='2026-01-01T10:00:03', device='sensor-C', reading=19.75)],
 [ParseFailure(raw_value='2026-01-01T10:00:01|sensor-B|bad', exception_type='ValueError', message="could not convert string to float: 'bad'"),
  ParseFailure(raw_value='missing|fields', exception_type='ValueError', message='expected 3 fields, received 2')])

## What this problem teaches

A useful pipeline stage should have one clear responsibility.

Here:

- `parse_sensor_line` understands the text format;
- `parsing_stage` understands success and failure routing;
- `list_sink` stores results.

This separation makes all three pieces reusable.


# Problem 2 — Stateful Anomaly Detection with Welford's Algorithm

Now we will build a stateful stage.

The stage receives numeric measurements and identifies unusually large deviations from the running mean.

A naive implementation could store all previous values, but that would use unbounded memory.

Instead, we will update the running mean and variance incrementally using Welford's algorithm.


## Step 1 — Define the output record

For every value, we want to know:

- the input value;
- how many values have been observed;
- the previous mean;
- the previous standard deviation;
- the resulting z-score;
- whether the value is considered anomalous.

We compare the new value against statistics computed **before** adding it.  
This avoids allowing the candidate value to immediately dilute its own anomaly score.


In [9]:
@dataclass(frozen=True)
class AnomalyResult:
    value: float
    previous_count: int
    previous_mean: Optional[float]
    previous_stddev: Optional[float]
    z_score: Optional[float]
    is_anomaly: bool


## Step 2 — Review the running-statistics update

Welford's algorithm maintains:

- `count`;
- `mean`;
- `m2`, the sum of squared deviations needed for variance.

For sample variance:

```python
variance = m2 / (count - 1)
```

when at least two values are available.


In [10]:
@dataclass
class RunningStats:
    count: int = 0
    mean: float = 0.0
    m2: float = 0.0

    def sample_stddev(self) -> Optional[float]:
        if self.count < 2:
            return None
        return sqrt(self.m2 / (self.count - 1))

    def update(self, value: float) -> None:
        self.count += 1
        delta = value - self.mean
        self.mean += delta / self.count
        delta_after_update = value - self.mean
        self.m2 += delta * delta_after_update


## Step 3 — Implement the anomaly detector

The detector needs a warm-up period.

Before enough historical values are available, we forward the result but do not mark anomalies.


In [11]:
@autostart
def anomaly_detector(
    target: Generator[Any, Any, Any],
    *,
    z_threshold: float = 3.0,
    minimum_history: int = 5,
):
    if z_threshold <= 0:
        raise ValueError("z_threshold must be positive")
    if minimum_history < 2:
        raise ValueError("minimum_history must be at least 2")

    stats = RunningStats()

    try:
        while True:
            value = float((yield))

            previous_count = stats.count
            previous_mean = stats.mean if stats.count else None
            previous_stddev = stats.sample_stddev()

            z_score: Optional[float] = None
            is_anomaly = False

            if (
                previous_count >= minimum_history
                and previous_stddev is not None
                and previous_stddev > 0
            ):
                z_score = (value - stats.mean) / previous_stddev
                is_anomaly = abs(z_score) >= z_threshold

            target.send(
                AnomalyResult(
                    value=value,
                    previous_count=previous_count,
                    previous_mean=previous_mean,
                    previous_stddev=previous_stddev,
                    z_score=z_score,
                    is_anomaly=is_anomaly,
                )
            )

            stats.update(value)
    finally:
        target.close()


## Step 4 — Test a normal sequence followed by a spike

The first five values establish a stable history.  
The value `100` should then be detected as anomalous.


In [12]:
anomaly_results: List[AnomalyResult] = []

anomaly_pipeline = anomaly_detector(
    list_sink(anomaly_results),
    z_threshold=3.0,
    minimum_history=5,
)

push_all([10, 11, 9, 10, 10, 100], anomaly_pipeline)

assert anomaly_results[-1].is_anomaly is True
assert anomaly_results[-1].z_score is not None

anomaly_results


[AnomalyResult(value=10.0, previous_count=0, previous_mean=None, previous_stddev=None, z_score=None, is_anomaly=False),
 AnomalyResult(value=11.0, previous_count=1, previous_mean=10.0, previous_stddev=None, z_score=None, is_anomaly=False),
 AnomalyResult(value=9.0, previous_count=2, previous_mean=10.5, previous_stddev=0.7071067811865476, z_score=None, is_anomaly=False),
 AnomalyResult(value=10.0, previous_count=3, previous_mean=10.0, previous_stddev=1.0, z_score=None, is_anomaly=False),
 AnomalyResult(value=10.0, previous_count=4, previous_mean=10.0, previous_stddev=0.816496580927726, z_score=None, is_anomaly=False),
 AnomalyResult(value=100.0, previous_count=5, previous_mean=10.0, previous_stddev=0.7071067811865476, z_score=127.27922061357854, is_anomaly=True)]

## Important limitation

This detector updates its model even after an anomaly.

That may be correct or incorrect depending on the application.

Alternative policies include:

- never learn from anomalies;
- learn from them only after review;
- cap extreme values before updating;
- maintain separate short-term and long-term models.


# Problem 3 — Sessionization by Inactivity Gap

Many event-processing systems group events into sessions.

A new session begins when the time gap between consecutive events for the same user exceeds a configured threshold.

We will process events in arrival order and maintain state independently for each user.


## Step 1 — Define the event and output structures

To keep the exercise deterministic, timestamps are represented as integer seconds.


In [13]:
@dataclass(frozen=True)
class UserEvent:
    user_id: str
    timestamp: int
    action: str


@dataclass(frozen=True)
class SessionizedEvent:
    user_id: str
    session_number: int
    timestamp: int
    action: str
    starts_new_session: bool


## Step 2 — Identify the required state

For each user, the stage needs to remember:

- the timestamp of the last event;
- the current session number.

The state dictionaries grow with the number of distinct users.  
A real long-running service may also need eviction for inactive users.


In [14]:
@autostart
def sessionize(
    inactivity_gap: int,
    target: Generator[Any, Any, Any],
):
    if inactivity_gap < 0:
        raise ValueError("inactivity_gap must be non-negative")

    last_timestamp: Dict[str, int] = {}
    session_number: Dict[str, int] = defaultdict(int)

    try:
        while True:
            event: UserEvent = yield

            previous_timestamp = last_timestamp.get(event.user_id)

            starts_new_session = (
                previous_timestamp is None
                or event.timestamp - previous_timestamp > inactivity_gap
            )

            if starts_new_session:
                session_number[event.user_id] += 1

            last_timestamp[event.user_id] = event.timestamp

            target.send(
                SessionizedEvent(
                    user_id=event.user_id,
                    session_number=session_number[event.user_id],
                    timestamp=event.timestamp,
                    action=event.action,
                    starts_new_session=starts_new_session,
                )
            )
    finally:
        target.close()


## Step 3 — Push interleaved events

The events for Alice and Bob are interleaved.

The stage must still maintain independent session state for each user.


In [15]:
sessionized_events: List[SessionizedEvent] = []

session_pipeline = sessionize(
    inactivity_gap=30,
    target=list_sink(sessionized_events),
)

events = [
    UserEvent("alice", 0, "open"),
    UserEvent("bob", 5, "open"),
    UserEvent("alice", 10, "click"),
    UserEvent("bob", 50, "purchase"),
    UserEvent("alice", 45, "close"),
]

push_all(events, session_pipeline)

assert [event.session_number for event in sessionized_events] == [1, 1, 1, 2, 2]
assert [event.starts_new_session for event in sessionized_events] == [
    True, True, False, True, True
]

sessionized_events


[SessionizedEvent(user_id='alice', session_number=1, timestamp=0, action='open', starts_new_session=True),
 SessionizedEvent(user_id='bob', session_number=1, timestamp=5, action='open', starts_new_session=True),
 SessionizedEvent(user_id='alice', session_number=1, timestamp=10, action='click', starts_new_session=False),
 SessionizedEvent(user_id='bob', session_number=2, timestamp=50, action='purchase', starts_new_session=True),
 SessionizedEvent(user_id='alice', session_number=2, timestamp=45, action='close', starts_new_session=True)]

## Ordering assumption

This solution assumes that events for each user arrive in timestamp order.

If late events can arrive, the logic becomes more complicated.  
Possible solutions include buffering, watermarking, or reprocessing.


# Problem 4 — Time-Bucket Aggregation with Explicit Flush

We now want to aggregate events into fixed time buckets.

For example, with a bucket width of 60 seconds:

- timestamps `0` through `59` belong to bucket `0`;
- timestamps `60` through `119` belong to bucket `60`;
- and so on.

The stage will emit a summary when it sees an event belonging to a later bucket.


## Step 1 — Define the summary

The summary stores:

- the bucket start;
- the number of values;
- the total;
- the average.


In [16]:
@dataclass(frozen=True)
class BucketSummary:
    bucket_start: int
    count: int
    total: float
    average: float


## Step 2 — Introduce an explicit flush command

A special command lets us emit the currently open bucket without closing the entire pipeline.

Using a dedicated type is clearer than using a normal value such as `None`.


In [17]:
@dataclass(frozen=True)
class FlushBucket:
    pass


## Step 3 — Implement the bucket stage

This implementation assumes timestamps arrive in non-decreasing order.

When the stage moves to a later bucket, it emits the previous bucket first.


In [18]:
@autostart
def bucket_aggregate(
    bucket_width: int,
    target: Generator[Any, Any, Any],
):
    if bucket_width <= 0:
        raise ValueError("bucket_width must be positive")

    current_bucket: Optional[int] = None
    count = 0
    total = 0.0

    def emit_current_bucket() -> None:
        nonlocal count, total

        if current_bucket is not None and count:
            target.send(
                BucketSummary(
                    bucket_start=current_bucket,
                    count=count,
                    total=total,
                    average=total / count,
                )
            )

        count = 0
        total = 0.0

    try:
        while True:
            message = yield

            if isinstance(message, FlushBucket):
                emit_current_bucket()
                current_bucket = None
                continue

            timestamp, raw_value = message
            value = float(raw_value)
            bucket_start = (timestamp // bucket_width) * bucket_width

            if current_bucket is None:
                current_bucket = bucket_start
            elif bucket_start < current_bucket:
                raise ValueError("timestamps must be non-decreasing")
            elif bucket_start > current_bucket:
                emit_current_bucket()
                current_bucket = bucket_start

            count += 1
            total += value
    finally:
        emit_current_bucket()
        target.close()


## Step 4 — Verify normal rollover and manual flushing


In [19]:
bucket_summaries: List[BucketSummary] = []
bucket_pipeline = bucket_aggregate(60, list_sink(bucket_summaries))

bucket_pipeline.send((5, 10))
bucket_pipeline.send((20, 20))
bucket_pipeline.send((61, 30))   # closes bucket 0
bucket_pipeline.send((70, 50))
bucket_pipeline.send(FlushBucket())
bucket_pipeline.send((130, 40))
bucket_pipeline.close()

assert bucket_summaries == [
    BucketSummary(0, 2, 30.0, 15.0),
    BucketSummary(60, 2, 80.0, 40.0),
    BucketSummary(120, 1, 40.0, 40.0),
]

bucket_summaries


[BucketSummary(bucket_start=0, count=2, total=30.0, average=15.0),
 BucketSummary(bucket_start=60, count=2, total=80.0, average=40.0),
 BucketSummary(bucket_start=120, count=1, total=40.0, average=40.0)]

## Empty buckets

This implementation emits only buckets that contain data.

If empty buckets must also be emitted, the rollover logic would need to generate summaries for every missing interval.


# Problem 5 — Retry Transient Failures Before Using a Dead-Letter Queue

Sometimes a transformation fails temporarily.

Examples include:

- a short-lived network failure;
- a service returning a temporary error;
- a resource being briefly unavailable.

We will create a stage that retries only explicitly identified transient exceptions.


## Step 1 — Define a transient exception and failure record


In [20]:
class TransientError(RuntimeError):
    """Represents a failure that may succeed on a later attempt."""


@dataclass(frozen=True)
class RetryFailure:
    item: Any
    attempts: int
    message: str


## Step 2 — Implement retry behavior

The stage will:

1. call the operation;
2. retry when `TransientError` is raised;
3. stop after `max_attempts`;
4. send exhausted items to a failure target;
5. allow unexpected exceptions to propagate.

No sleeping is used here because the notebook should execute quickly and deterministically.


In [21]:
@autostart
def retrying_map(
    operation: Callable[[Any], Any],
    success_target: Generator[Any, Any, Any],
    failure_target: Generator[Any, Any, Any],
    *,
    max_attempts: int = 3,
):
    if max_attempts <= 0:
        raise ValueError("max_attempts must be positive")

    try:
        while True:
            item = yield
            last_error: Optional[TransientError] = None

            for attempt in range(1, max_attempts + 1):
                try:
                    result = operation(item)
                except TransientError as exc:
                    last_error = exc
                else:
                    success_target.send(result)
                    break
            else:
                failure_target.send(
                    RetryFailure(
                        item=item,
                        attempts=max_attempts,
                        message=str(last_error),
                    )
                )
    finally:
        success_target.close()
        failure_target.close()


## Step 3 — Create a deterministic flaky operation

The operation below fails a known number of times for each input.

That makes retry behavior easy to test.


In [22]:
attempt_counter: Dict[str, int] = defaultdict(int)
failures_before_success = {
    "A": 0,
    "B": 2,
    "C": 5,
}


def flaky_lookup(key: str) -> str:
    attempt_counter[key] += 1

    if attempt_counter[key] <= failures_before_success[key]:
        raise TransientError(f"temporary failure for {key}")

    return f"value-for-{key}"


## Step 4 — Run the retry pipeline

With three attempts:

- `A` succeeds immediately;
- `B` succeeds on the third attempt;
- `C` is sent to the failure sink.


In [23]:
retry_successes: List[str] = []
retry_failures: List[RetryFailure] = []

retry_pipeline = retrying_map(
    flaky_lookup,
    list_sink(retry_successes),
    list_sink(retry_failures),
    max_attempts=3,
)

push_all(["A", "B", "C"], retry_pipeline)

assert retry_successes == ["value-for-A", "value-for-B"]
assert retry_failures == [
    RetryFailure("C", 3, "temporary failure for C")
]
assert attempt_counter == {"A": 1, "B": 3, "C": 3}

retry_successes, retry_failures, dict(attempt_counter)


(['value-for-A', 'value-for-B'],
 [RetryFailure(item='C', attempts=3, message='temporary failure for C')],
 {'A': 1, 'B': 3, 'C': 3})

## Retry design warning

Retries can make a problem worse when:

- the operation is not idempotent;
- the downstream service is overloaded;
- there is no delay or backoff;
- every item retries at the same time.

Real retry systems often add exponential backoff, jitter, limits, and circuit breakers.


# Problem 6 — A Circuit Breaker Stage

A circuit breaker prevents repeated calls to an operation that is currently failing.

We will use three states:

- **closed**: calls are allowed;
- **open**: calls are blocked;
- **half-open**: one trial call is allowed.

Because this is a deterministic notebook, state changes are controlled with command messages rather than wall-clock timers.


## Step 1 — Define commands and observations


In [24]:
@dataclass(frozen=True)
class ResetCircuit:
    """Move an open circuit into half-open state."""


@dataclass(frozen=True)
class CircuitObservation:
    item: Any
    state_before: str
    outcome: str


## Step 2 — Decide the opening rule

The breaker opens after a configured number of consecutive transient failures.

Successful calls reset the failure counter.

While open, ordinary data is not sent to the protected operation.


In [25]:
@autostart
def circuit_breaker(
    operation: Callable[[Any], Any],
    success_target: Generator[Any, Any, Any],
    observation_target: Generator[Any, Any, Any],
    *,
    failure_threshold: int = 2,
):
    if failure_threshold <= 0:
        raise ValueError("failure_threshold must be positive")

    state = "closed"
    consecutive_failures = 0

    try:
        while True:
            message = yield

            if isinstance(message, ResetCircuit):
                if state == "open":
                    state = "half-open"
                continue

            item = message
            state_before = state

            if state == "open":
                observation_target.send(
                    CircuitObservation(item, state_before, "blocked")
                )
                continue

            try:
                result = operation(item)
            except TransientError:
                consecutive_failures += 1
                observation_target.send(
                    CircuitObservation(item, state_before, "failure")
                )

                if state == "half-open" or consecutive_failures >= failure_threshold:
                    state = "open"
            else:
                success_target.send(result)
                observation_target.send(
                    CircuitObservation(item, state_before, "success")
                )
                consecutive_failures = 0
                state = "closed"
    finally:
        success_target.close()
        observation_target.close()


## Step 3 — Test state transitions

The protected operation fails for values beginning with `"bad"`.


In [26]:
def guarded_operation(value: str) -> str:
    if value.startswith("bad"):
        raise TransientError(value)
    return value.upper()


circuit_successes: List[str] = []
circuit_observations: List[CircuitObservation] = []

breaker = circuit_breaker(
    guarded_operation,
    list_sink(circuit_successes),
    list_sink(circuit_observations),
    failure_threshold=2,
)

breaker.send("bad-1")      # closed -> failure
breaker.send("bad-2")      # closed -> failure -> open
breaker.send("good-1")     # blocked
breaker.send(ResetCircuit())
breaker.send("good-2")     # half-open -> success -> closed
breaker.send("good-3")     # normal success
breaker.close()

assert circuit_successes == ["GOOD-2", "GOOD-3"]
assert [item.outcome for item in circuit_observations] == [
    "failure", "failure", "blocked", "success", "success"
]

circuit_successes, circuit_observations


(['GOOD-2', 'GOOD-3'],
 [CircuitObservation(item='bad-1', state_before='closed', outcome='failure'),
  CircuitObservation(item='bad-2', state_before='closed', outcome='failure'),
  CircuitObservation(item='good-1', state_before='open', outcome='blocked'),
  CircuitObservation(item='good-2', state_before='half-open', outcome='success'),
  CircuitObservation(item='good-3', state_before='closed', outcome='success')])

## Why commands are useful

A command object is clearly distinguishable from ordinary stream data.

It also makes the control protocol visible in the code:

```python
breaker.send(ResetCircuit())
```

This is easier to reason about than overloading a value such as `None` or a magic string.


# Problem 7 — Pause, Resume, and Dynamic Threshold Configuration

Now we will create a filter that can be controlled while data is flowing.

The stage will support three command types:

- pause;
- resume;
- update the threshold.

Paused values will not be forwarded.


In [27]:
@dataclass(frozen=True)
class Pause:
    pass


@dataclass(frozen=True)
class Resume:
    pass


@dataclass(frozen=True)
class SetMinimum:
    value: float


## Step 1 — Implement the controlled filter

The current control state lives inside the coroutine.

Ordinary numeric values are forwarded only when:

- the stage is active;
- the value is at least the current minimum.


In [28]:
@autostart
def controlled_minimum_filter(
    initial_minimum: float,
    target: Generator[Any, Any, Any],
):
    minimum = float(initial_minimum)
    paused = False

    try:
        while True:
            message = yield

            if isinstance(message, Pause):
                paused = True
                continue

            if isinstance(message, Resume):
                paused = False
                continue

            if isinstance(message, SetMinimum):
                minimum = float(message.value)
                continue

            value = float(message)

            if not paused and value >= minimum:
                target.send(value)
    finally:
        target.close()


## Step 2 — Interleave control messages and data


In [29]:
controlled_output: List[float] = []

controlled_pipeline = controlled_minimum_filter(
    10,
    list_sink(controlled_output),
)

controlled_pipeline.send(5)
controlled_pipeline.send(12)
controlled_pipeline.send(Pause())
controlled_pipeline.send(100)
controlled_pipeline.send(SetMinimum(50))
controlled_pipeline.send(Resume())
controlled_pipeline.send(40)
controlled_pipeline.send(75)
controlled_pipeline.close()

assert controlled_output == [12.0, 75.0]
controlled_output


[12.0, 75.0]

## Control-message semantics

This design intentionally drops data received while paused.

Another valid policy would be to buffer paused data, but that creates new questions:

- How large can the buffer become?
- What happens if the process stops?
- Should old or new values be dropped when the buffer is full?


# Problem 8 — Join Two Logical Streams by Correlation ID

A single coroutine can receive messages representing two logical streams.

We will join:

- request messages;
- response messages.

Both message types contain the same correlation ID.

The joined result is emitted once both sides are available.


## Step 1 — Define the message protocol


In [30]:
@dataclass(frozen=True)
class RequestMessage:
    correlation_id: str
    payload: Any


@dataclass(frozen=True)
class ResponseMessage:
    correlation_id: str
    payload: Any


@dataclass(frozen=True)
class JoinedExchange:
    correlation_id: str
    request: Any
    response: Any


## Step 2 — Determine the buffering strategy

A request may arrive first, or a response may arrive first.

Therefore, we maintain two dictionaries:

- unmatched requests;
- unmatched responses.

When a matching counterpart exists, both buffered values are removed and one joined record is emitted.


In [31]:
@autostart
def join_requests_and_responses(
    target: Generator[Any, Any, Any],
):
    requests: Dict[str, Any] = {}
    responses: Dict[str, Any] = {}

    try:
        while True:
            message = yield

            if isinstance(message, RequestMessage):
                correlation_id = message.correlation_id

                if correlation_id in requests:
                    raise ValueError(
                        f"duplicate request for {correlation_id!r}"
                    )

                if correlation_id in responses:
                    response_payload = responses.pop(correlation_id)
                    target.send(
                        JoinedExchange(
                            correlation_id,
                            message.payload,
                            response_payload,
                        )
                    )
                else:
                    requests[correlation_id] = message.payload

            elif isinstance(message, ResponseMessage):
                correlation_id = message.correlation_id

                if correlation_id in responses:
                    raise ValueError(
                        f"duplicate response for {correlation_id!r}"
                    )

                if correlation_id in requests:
                    request_payload = requests.pop(correlation_id)
                    target.send(
                        JoinedExchange(
                            correlation_id,
                            request_payload,
                            message.payload,
                        )
                    )
                else:
                    responses[correlation_id] = message.payload

            else:
                raise TypeError(
                    f"unsupported message type: {type(message).__name__}"
                )
    finally:
        target.close()


## Step 3 — Test both arrival orders


In [32]:
joined_exchanges: List[JoinedExchange] = []

join_pipeline = join_requests_and_responses(
    list_sink(joined_exchanges)
)

join_pipeline.send(RequestMessage("A", {"path": "/users"}))
join_pipeline.send(ResponseMessage("B", {"status": 200}))
join_pipeline.send(ResponseMessage("A", {"status": 404}))
join_pipeline.send(RequestMessage("B", {"path": "/orders"}))
join_pipeline.close()

assert joined_exchanges == [
    JoinedExchange(
        "A",
        {"path": "/users"},
        {"status": 404},
    ),
    JoinedExchange(
        "B",
        {"path": "/orders"},
        {"status": 200},
    ),
]

joined_exchanges


[JoinedExchange(correlation_id='A', request={'path': '/users'}, response={'status': 404}),
 JoinedExchange(correlation_id='B', request={'path': '/orders'}, response={'status': 200})]

## Memory and expiration

This implementation retains unmatched messages until the pipeline closes.

A real system normally needs an expiration policy because some counterparts may never arrive.

Possible approaches include:

- maximum age;
- maximum number of buffered entries;
- explicit cancellation messages;
- periodic cleanup commands;
- external durable storage.


# Problem 9 — A Multi-Stage Tutorial Pipeline

We will now combine several ideas into one realistic pipeline.

The input is a stream of API latency records:

```python
{
    "service": "Billing",
    "latency_ms": "125",
    "timestamp": 10
}
```

The completed pipeline will:

1. normalize the record;
2. route malformed records to an error sink;
3. keep only positive latency values;
4. detect anomalies;
5. sessionize activity by service;
6. store final enriched records.

The purpose is not just to produce a result.  
It is to demonstrate how larger pipelines can be built from small stages.


## Step 1 — Define normalized and enriched records


In [33]:
@dataclass(frozen=True)
class ApiLatency:
    service: str
    latency_ms: float
    timestamp: int


@dataclass(frozen=True)
class EnrichedLatency:
    service: str
    latency_ms: float
    timestamp: int
    session_number: int
    is_anomaly: bool


## Step 2 — Normalize raw dictionaries

Normalization includes:

- trimming and lowercasing the service name;
- parsing latency as `float`;
- parsing timestamp as `int`.


In [34]:
def normalize_latency_record(raw: Mapping[str, Any]) -> ApiLatency:
    service = str(raw["service"]).strip().lower()
    latency_ms = float(raw["latency_ms"])
    timestamp = int(raw["timestamp"])

    if not service:
        raise ValueError("service must not be empty")

    return ApiLatency(service, latency_ms, timestamp)


## Step 3 — Create a generic predicate stage

This stage sends rejected values to a separate sink with a reason.


In [35]:
@dataclass(frozen=True)
class RejectedValue:
    value: Any
    reason: str


@autostart
def predicate_stage(
    predicate: Callable[[Any], bool],
    accepted_target: Generator[Any, Any, Any],
    rejected_target: Generator[Any, Any, Any],
    *,
    reason: str,
):
    try:
        while True:
            value = yield

            if predicate(value):
                accepted_target.send(value)
            else:
                rejected_target.send(RejectedValue(value, reason))
    finally:
        accepted_target.close()
        rejected_target.close()


## Step 4 — Build a per-service anomaly detector

The earlier anomaly detector maintained one global statistical model.

This version maintains separate running statistics for each service.


In [36]:
@dataclass(frozen=True)
class ServiceAnomaly:
    record: ApiLatency
    is_anomaly: bool


@autostart
def per_service_anomaly_detector(
    target: Generator[Any, Any, Any],
    *,
    z_threshold: float = 3.0,
    minimum_history: int = 3,
):
    stats_by_service: Dict[str, RunningStats] = defaultdict(RunningStats)

    try:
        while True:
            record: ApiLatency = yield
            stats = stats_by_service[record.service]
            stddev = stats.sample_stddev()

            is_anomaly = False

            if (
                stats.count >= minimum_history
                and stddev is not None
                and stddev > 0
            ):
                z_score = (record.latency_ms - stats.mean) / stddev
                is_anomaly = abs(z_score) >= z_threshold

            target.send(ServiceAnomaly(record, is_anomaly))
            stats.update(record.latency_ms)
    finally:
        target.close()


## Step 5 — Add service session numbers

This stage resembles the earlier sessionizer, but it consumes `ServiceAnomaly` objects and emits `EnrichedLatency` objects.


In [37]:
@autostart
def enrich_with_service_session(
    inactivity_gap: int,
    target: Generator[Any, Any, Any],
):
    last_timestamp: Dict[str, int] = {}
    session_number: Dict[str, int] = defaultdict(int)

    try:
        while True:
            anomaly: ServiceAnomaly = yield
            record = anomaly.record
            previous = last_timestamp.get(record.service)

            if previous is None or record.timestamp - previous > inactivity_gap:
                session_number[record.service] += 1

            last_timestamp[record.service] = record.timestamp

            target.send(
                EnrichedLatency(
                    service=record.service,
                    latency_ms=record.latency_ms,
                    timestamp=record.timestamp,
                    session_number=session_number[record.service],
                    is_anomaly=anomaly.is_anomaly,
                )
            )
    finally:
        target.close()


## Step 6 — Assemble the final pipeline

Reading from the outside inward:

```text
raw input
    → parser
    → positive-latency validation
    → per-service anomaly detection
    → service session enrichment
    → result sink
```


In [38]:
final_results: List[EnrichedLatency] = []
final_parse_failures: List[ParseFailure] = []
final_validation_failures: List[RejectedValue] = []

final_pipeline = parsing_stage(
    normalize_latency_record,
    predicate_stage(
        lambda record: record.latency_ms > 0,
        per_service_anomaly_detector(
            enrich_with_service_session(
                30,
                list_sink(final_results),
            ),
            z_threshold=3.0,
            minimum_history=3,
        ),
        list_sink(final_validation_failures),
        reason="latency must be positive",
    ),
    list_sink(final_parse_failures),
)


## Step 7 — Push realistic test data

The billing service establishes a stable baseline near 100 ms, then receives a large spike.

The search service has a separate model and separate session state.


In [39]:
latency_input = [
    {"service": "Billing", "latency_ms": "100", "timestamp": 0},
    {"service": "Billing", "latency_ms": "102", "timestamp": 5},
    {"service": "Search", "latency_ms": "50", "timestamp": 6},
    {"service": "Billing", "latency_ms": "98", "timestamp": 10},
    {"service": "Billing", "latency_ms": "101", "timestamp": 15},
    {"service": "Billing", "latency_ms": "500", "timestamp": 20},
    {"service": "Search", "latency_ms": "-1", "timestamp": 50},
    {"service": "Search", "latency_ms": "bad", "timestamp": 55},
    {"service": "Search", "latency_ms": "55", "timestamp": 100},
]

push_all(latency_input, final_pipeline)

assert len(final_results) == 7
assert len(final_validation_failures) == 1
assert len(final_parse_failures) == 1

billing_spike = next(
    item
    for item in final_results
    if item.service == "billing" and item.latency_ms == 500
)
assert billing_spike.is_anomaly is True

search_events = [
    item for item in final_results if item.service == "search"
]
assert [item.session_number for item in search_events] == [1, 2]

final_results, final_validation_failures, final_parse_failures


([EnrichedLatency(service='billing', latency_ms=100.0, timestamp=0, session_number=1, is_anomaly=False),
  EnrichedLatency(service='billing', latency_ms=102.0, timestamp=5, session_number=1, is_anomaly=False),
  EnrichedLatency(service='search', latency_ms=50.0, timestamp=6, session_number=1, is_anomaly=False),
  EnrichedLatency(service='billing', latency_ms=98.0, timestamp=10, session_number=1, is_anomaly=False),
  EnrichedLatency(service='billing', latency_ms=101.0, timestamp=15, session_number=1, is_anomaly=False),
  EnrichedLatency(service='billing', latency_ms=500.0, timestamp=20, session_number=1, is_anomaly=True),
  EnrichedLatency(service='search', latency_ms=55.0, timestamp=100, session_number=2, is_anomaly=False)],
 [RejectedValue(value=ApiLatency(service='search', latency_ms=-1.0, timestamp=50), reason='latency must be positive')],
 [ParseFailure(raw_value={'service': 'Search', 'latency_ms': 'bad', 'timestamp': 55}, exception_type='ValueError', message="could not convert str

# Problem 10 — Safe Shutdown and Partial State

Closing a coroutine raises `GeneratorExit` at its current suspension point.

A stage that owns downstream resources should normally close them in a `finally` block.

However, stateful stages may also need to emit partial state before shutting down.

We will build a fixed-count summary stage that emits every `n` values and flushes a partial summary when closed.


In [40]:
@dataclass(frozen=True)
class CountSummary:
    values: Tuple[float, ...]
    total: float
    average: float


@autostart
def summarize_every(
    count: int,
    target: Generator[Any, Any, Any],
):
    if count <= 0:
        raise ValueError("count must be positive")

    buffer: List[float] = []

    def emit() -> None:
        if buffer:
            snapshot = tuple(buffer)
            target.send(
                CountSummary(
                    values=snapshot,
                    total=sum(snapshot),
                    average=sum(snapshot) / len(snapshot),
                )
            )
            buffer.clear()

    try:
        while True:
            value = float((yield))
            buffer.append(value)

            if len(buffer) == count:
                emit()
    finally:
        emit()
        target.close()


## Test close-time flushing

The final group contains only two values, but it should still be emitted during `.close()`.


In [41]:
count_summaries: List[CountSummary] = []
summary_pipeline = summarize_every(3, list_sink(count_summaries))

for value in [1, 2, 3, 4, 5]:
    summary_pipeline.send(value)

summary_pipeline.close()

assert count_summaries == [
    CountSummary((1.0, 2.0, 3.0), 6.0, 2.0),
    CountSummary((4.0, 5.0), 9.0, 4.5),
]

count_summaries


[CountSummary(values=(1.0, 2.0, 3.0), total=6.0, average=2.0),
 CountSummary(values=(4.0, 5.0), total=9.0, average=4.5)]

# Further Practice Problems

The following problems are intentionally left without full code so that you can extend the notebook.

## Practice A — Duplicate suppression with expiration

Build a stage that suppresses duplicate event IDs for the next `n` received events, then allows the ID again.

Consider:

- how to remove expired IDs efficiently;
- whether the window counts all received events or only accepted events;
- whether one ID may appear more than once inside the window.

## Practice B — Per-key rate tracking

For each key, compute events per logical time interval.

Consider:

- whether timestamps are ordered;
- how long inactive keys remain in memory;
- what to emit for keys with no events.

## Practice C — Priority buffering

Receive `(priority, value)` pairs and emit the highest-priority value when a `DrainOne` command arrives.

Consider:

- stable ordering among equal priorities;
- maximum queue size;
- shutdown behavior.

## Practice D — Rolling median

Implement a fixed-size rolling median stage.

A simple sorted-list solution is acceptable first.  
Then investigate a two-heap design for better asymptotic performance.


# Best-Practice Checklist

When designing a generator-coroutine pipeline, ask the following questions.

## Data contract

- What type does the stage receive?
- What type does it emit?
- Can it emit zero, one, or many outputs for one input?

## State

- What state is retained?
- Can that state grow without bound?
- Does state need expiration, persistence, or inspection?

## Failure policy

- Which exceptions represent bad data?
- Which exceptions represent programming errors?
- Should failures propagate, retry, or go to a separate sink?

## Shutdown

- Who owns each downstream target?
- Should buffered values be flushed?
- Can closing one stage accidentally close a shared target?

## Ordering and blocking

- Is input order preserved?
- Can a slow stage block the full pipeline?
- Would queues or `asyncio` be a better fit?

## Control protocol

- Are commands represented by explicit command classes?
- Can command values collide with normal data?
- Are pause, flush, reset, and reconfiguration semantics documented?


# Closing Perspective

Push-based coroutine pipelines are compact because `yield` performs two jobs:

1. it suspends the stage while waiting for input;
2. it receives the next value sent by the upstream caller.

The basic pattern is:

```python
while True:
    item = yield
    result = process(item)
    target.send(result)
```

Advanced pipelines add:

- validation;
- state;
- branching;
- commands;
- retries;
- buffering;
- graceful shutdown;
- explicit error channels.

The most important lesson is not the syntax.  
It is the discipline of defining clear contracts between stages.
